<a href="https://colab.research.google.com/github/ErnestoTrader/reaper-helper-2tb/blob/main/BULK_SMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import time
import os
import json
import re  # Added for aggressive number cleaning
from google.colab import files

# ==========================================
# --- CONFIGURATION ---
# ==========================================
# SMSGate API endpoint for Cloud Server
SMSGATE_URL = "https://api.sms-gate.app/3rdparty/v1/messages"

# Replace with your Cloud Server credentials from the app
USERNAME = "2XHOS1"
PASSWORD = "7yaif7rxdcufvc"

# Your message to the farmers
MESSAGE_TEXT = """Dear  farmer,
We are pleased to inform you that your day for fertilizer collection has been scheduled for this Thursday 19/03/2026 @ 7:30am. Welcome.
NAKURU NCPB"""

# File to keep track of our progress
PROGRESS_FILE = "sms_progress.json"
# ==========================================

def format_kenyan_number(phone):
    """Formats phone numbers strictly to E.164 international format (+254...)."""
    # 1. Convert to string
    num_str = str(phone)

    # 2. Aggressively remove EVERYTHING that is not a digit (0-9)
    # This deletes invisible spaces, dashes, parentheses, decimals, etc.
    clean_digits = re.sub(r'\D', '', num_str)

    # 3. Apply the Kenyan formatting rules and add the strict '+' sign
    if clean_digits.startswith('0') and len(clean_digits) == 10:
        return '+254' + clean_digits[1:]

    elif len(clean_digits) == 9 and (clean_digits.startswith('7') or clean_digits.startswith('1')):
        return '+254' + clean_digits

    elif clean_digits.startswith('254') and len(clean_digits) >= 12:
        return '+' + clean_digits

    # If it's something unexpected, just return it with a plus
    return '+' + clean_digits

def upload_and_read_data():
    """Handles the uploading and reading of the Excel file."""
    print("Please upload your Excel file containing the farmers' numbers.")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return None

    filename = list(uploaded.keys())[0]

    try:
        df = pd.read_excel(filename)

        # Check for the 'Phone' column
        if 'Phone' not in df.columns:
            print("Error: Your Excel file must have a column named exactly 'Phone'.")
            return None

        # Drop any rows where the 'Phone' column is completely blank
        df = df.dropna(subset=['Phone']).reset_index(drop=True)
        return df

    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None

def send_sms(phone, message):
    """Sends a single SMS using the newest SMSGate API standard."""
    payload = {
        "textMessage": {
            "text": message
        },
        "phoneNumbers": [str(phone)]
    }

    try:
        response = requests.post(
            SMSGATE_URL,
            json=payload,
            auth=(USERNAME, PASSWORD),
            timeout=15
        )

        # 200, 201, and 202 are all successful API responses
        if response.status_code in [200, 201, 202]:
            return True
        else:
            print(f" [API Error {response.status_code}: {response.text}] ", end="")
            return False

    except requests.exceptions.RequestException:
        print(f" [Connection Error] ", end="")
        return False

def main():
    df = upload_and_read_data()
    if df is None:
        return

    total_records = len(df)
    failed_numbers = []
    start_index = 0

    # --- RESTART LOGIC ---
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            try:
                data = json.load(f)
                resume_choice = input(f"\nFound previous progress. You left off at index {data['last_index']}. Resume? (y/n): ")
                if resume_choice.lower() == 'y':
                    start_index = data['last_index'] + 1
                else:
                    print("Starting fresh...")
            except json.JSONDecodeError:
                print("Progress file corrupted. Starting fresh...")

    print(f"\nStarting SMS broadcast to {total_records - start_index} farmers...")
    print("To STOP the process gracefully, click the 'Stop' button (interrupt execution) on the left of the cell.\n")

    try:
        for index in range(start_index, total_records):
            raw_phone = df.loc[index, 'Phone']
            clean_number = format_kenyan_number(raw_phone)

            print(f"[{index + 1}/{total_records}] Sending SMS to {clean_number}...", end=" ")

            success = send_sms(clean_number, MESSAGE_TEXT)

            if success:
                print("Success!")
            else:
                print("FAILED.")
                failed_numbers.append(clean_number)

            # Save progress immediately after the attempt
            with open(PROGRESS_FILE, "w") as f:
                json.dump({"last_index": index}, f)

            # --- 2 SECOND DELAY ---
            time.sleep(2)

    except KeyboardInterrupt:
        print("\n\n🛑 Process manually stopped! Progress has been saved.")
        print("You can run this cell again to restart/resume from where you left off.")
        return

    # --- FINAL REPORT ---
    print("\n" + "="*30)
    print("📊 SMS BROADCAST COMPLETE")
    print("="*30)

    total_attempted = total_records - start_index
    print(f"Total Attempted in this run: {total_attempted}")
    print(f"Successful: {total_attempted - len(failed_numbers)}")
    print(f"Failed: {len(failed_numbers)}")

    if failed_numbers:
        print("\nList of failed numbers:")
        for num in failed_numbers:
            print(f"- {num}")

    # Clean up progress file since we finished successfully
    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

if __name__ == "__main__":
    main()

Please upload your Excel file containing the farmers' numbers.


Saving FARMERS LIST 19TH MARCH 2026.xlsx to FARMERS LIST 19TH MARCH 2026.xlsx

Starting SMS broadcast to 266 farmers...
To STOP the process gracefully, click the 'Stop' button (interrupt execution) on the left of the cell.

[1/266] Sending SMS to +254795514354... Success!
[2/266] Sending SMS to +254719888072... Success!
[3/266] Sending SMS to +254710277046... Success!
[4/266] Sending SMS to +254725935776... Success!
[5/266] Sending SMS to +254723884947... Success!
[6/266] Sending SMS to +254700584716... Success!
[7/266] Sending SMS to +254797621224... Success!
[8/266] Sending SMS to +254729975929... Success!
[9/266] Sending SMS to +254714423995... Success!
[10/266] Sending SMS to +254721680791... Success!
[11/266] Sending SMS to +254720842172... Success!
[12/266] Sending SMS to +254721504490... Success!
[13/266] Sending SMS to +254716918256... Success!
[14/266] Sending SMS to +254721282440... Success!
[15/266] Sending SMS to +254721819081... Success!
[16/266] Sending SMS to +2547282554